### الخلية 1 كانت تنصيب ال 
!pip install web3 pandas py-solc-x

### Cell 2 — الاستيراد وإنشاء المسارات الرسمية

In [3]:
# -*- coding: utf-8 -*-

from pathlib import Path
import os
import re
import json
import secrets

import pandas as pd
from web3 import Web3

# ============================================================
# المسار الرئيسي المعتمد لمرحلة إيصال التوصيات
# ============================================================

BASE_DELIVERY_DIR = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد"
)

CONTRACT_DIR = BASE_DELIVERY_DIR / "01_contract"
PYTHON_SCRIPTS_DIR = BASE_DELIVERY_DIR / "02_Python_scripts"
OUTPUTS_DIR = BASE_DELIVERY_DIR / "03_outputs"

CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
PYTHON_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DELIVERY_DIR:", BASE_DELIVERY_DIR)
print("CONTRACT_DIR:", CONTRACT_DIR)
print("PYTHON_SCRIPTS_DIR:", PYTHON_SCRIPTS_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)

BASE_DELIVERY_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد
CONTRACT_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\01_contract
PYTHON_SCRIPTS_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\02_Python_scripts
OUTPUTS_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\03_outputs


# الخطوة 2
هنا نكتب العقد من خلال البايثون

## Cell 3 — كود العقد الذكي كمرحلة مستقلة

هذه الخلية تحفظ كود العقد الذكي في:

01_contract\RecommendationDelivery.sol

هذا العقد يقوم بالآتي:

1. تسجيل العقدة الموثوقة A_k لكل مجتمع
2. تسجيل أعضاء كل مجتمع
3. منع تخزين Hash لتوصيات مستخدم غير عضو في المجتمع
4. منع أي جهة غير A_k من تخزين Hash توصيات ذلك المجتمع
5. تخزين Hash فقط وليس التوصيات الكاملة

In [4]:
SOLIDITY_CODE = r'''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/*
    RecommendationDelivery

    الهدف العام:
    - لا يخزن التوصيات الكاملة على البلوكشين.
    - يخزن فقط hash يمثل حزمة توصيات المستخدم.
    - يسجل العقدة الموثوقة A_k لكل مجتمع.
    - يسجل أعضاء كل مجتمع حتى لا يتم تخزين Hash لتوصيات مستخدم غير عضو.
    - يسمح فقط لـ A_k الخاصة بالمجتمع بتخزين Hash التوصيات.

    مفتاح التخزين:
    communityId + userAddress + version + topN

    ملاحظة منهجية:
    التوصيات الفعلية تبقى خارج البلوكشين في Payload JSON
    أو في قناة off-chain مثل API أو node-to-node communication.
*/

contract RecommendationDelivery {

    address public owner;

    struct RecommendationRecord {
        bytes32 recommendationHash;   // Hash of recommendation payload
        address issuer;               // Authorized node A_k that stored the hash
        uint256 timestamp;            // Block timestamp
        bool exists;                  // Record existence flag
    }

    // communityId => authorized node A_k
    mapping(uint256 => address) public authorizedNode;

    // communityId => userAddress => membership status
    mapping(uint256 => mapping(address => bool)) public isCommunityMember;

    // communityId => number of registered members
    mapping(uint256 => uint256) public communityMemberCount;

    /*
        records[communityId][user][version][topN] = RecommendationRecord
    */
    mapping(uint256 => mapping(address => mapping(uint256 => mapping(uint256 => RecommendationRecord)))) private records;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Only owner can perform this action");
        _;
    }

    modifier onlyAuthorizedNode(uint256 communityId) {
        require(
            msg.sender == authorizedNode[communityId],
            "Only authorized node can store recommendation hashes"
        );
        _;
    }

    /*
        تعيين العقدة الموثوقة A_k لمجتمع معين.
        هذا يتم من قبل owner بعد قراءة نتائج التصويت.
    */
    function setAuthorizedNode(
        uint256 communityId,
        address node
    ) external onlyOwner {
        require(node != address(0), "Invalid authorized node address");
        authorizedNode[communityId] = node;
    }

    /*
        تسجيل عضو واحد داخل مجتمع.
        العضوية هنا تستخدم userAddress وليس user_id.
    */
    function registerCommunityMember(
        uint256 communityId,
        address member
    ) external onlyOwner {
        require(member != address(0), "Invalid member address");

        if (!isCommunityMember[communityId][member]) {
            isCommunityMember[communityId][member] = true;
            communityMemberCount[communityId] += 1;
        }
    }

    /*
        تسجيل مجموعة أعضاء داخل مجتمع.
        هذا يقلل عدد المعاملات مقارنة بتسجيل كل عضو منفردًا.
    */
    function registerCommunityMembers(
        uint256 communityId,
        address[] calldata members
    ) external onlyOwner {
        for (uint256 i = 0; i < members.length; i++) {
            address member = members[i];
            require(member != address(0), "Invalid member address");

            if (!isCommunityMember[communityId][member]) {
                isCommunityMember[communityId][member] = true;
                communityMemberCount[communityId] += 1;
            }
        }
    }

    /*
        فحص هل النود عضو في المجتمع.
    */
    function checkCommunityMember(
        uint256 communityId,
        address member
    ) external view returns (bool) {
        return isCommunityMember[communityId][member];
    }

    /*
        تخزين Hash التوصيات فقط.

        شروط القبول:
        1. msg.sender هو A_k لذلك المجتمع.
        2. user عضو مسجل في المجتمع.
        3. السجل غير مخزن سابقًا لنفس communityId + user + version + topN.
    */
    function storeRecommendationHash(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN,
        bytes32 recommendationHash
    ) external onlyAuthorizedNode(communityId) {
        require(user != address(0), "Invalid user address");
        require(topN > 0, "Invalid topN value");
        require(recommendationHash != bytes32(0), "Invalid recommendation hash");

        require(
            isCommunityMember[communityId][user],
            "Target user is not a registered member of this community"
        );

        require(
            records[communityId][user][version][topN].exists == false,
            "Recommendation hash already exists for this key"
        );

        records[communityId][user][version][topN] = RecommendationRecord({
            recommendationHash: recommendationHash,
            issuer: msg.sender,
            timestamp: block.timestamp,
            exists: true
        });
    }

    /*
        فحص وجود سجل سابق.
    */
    function recordExists(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN
    ) external view returns (bool) {
        return records[communityId][user][version][topN].exists;
    }

    /*
        جلب الهاش المخزون.
    */
    function getRecommendationHash(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN
    ) external view returns (bytes32) {
        require(
            records[communityId][user][version][topN].exists,
            "Recommendation record not found"
        );

        return records[communityId][user][version][topN].recommendationHash;
    }

    /*
        مقارنة الهاش المحلي مع الهاش المخزون.
        هذه الدالة تتحقق من سلامة البيانات Integrity فقط.
        لا تتحقق من جودة التوصيات.
    */
    function verifyRecommendationHash(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN,
        bytes32 receivedHash
    ) external view returns (bool) {
        if (!records[communityId][user][version][topN].exists) {
            return false;
        }

        return records[communityId][user][version][topN].recommendationHash == receivedHash;
    }

    /*
        جلب تفاصيل السجل لغرض المراجعة والتدقيق.
    */
    function getRecommendationRecord(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN
    ) external view returns (
        bytes32 recommendationHash,
        address issuer,
        uint256 timestamp,
        bool exists
    ) {
        RecommendationRecord memory record = records[communityId][user][version][topN];

        return (
            record.recommendationHash,
            record.issuer,
            record.timestamp,
            record.exists
        );
    }
}
'''

SOLIDITY_FILE = CONTRACT_DIR / "RecommendationDelivery.sol"
SOLIDITY_FILE.write_text(SOLIDITY_CODE, encoding="utf-8")

print("Smart contract source saved to:")
print(SOLIDITY_FILE)

Smart contract source saved to:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\01_contract\RecommendationDelivery.sol


### الخطوة 3
### Cell 4 — الاتصال بـ Ganache CLI

In [5]:
RPC_URL = "http://127.0.0.1:7545"

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise RuntimeError(
        "Cannot connect to Ganache. تأكد أن Ganache CLI شغال على http://127.0.0.1:7545"
    )

accounts = w3.eth.accounts

if len(accounts) < 2:
    raise RuntimeError("Ganache must provide at least two accounts.")

OWNER_ACCOUNT = accounts[0]

print("Connected to Ganache:", w3.is_connected())
print("Chain ID:", w3.eth.chain_id)
print("Number of accounts:", len(accounts))
print("Owner Account:", OWNER_ACCOUNT)

print("\nFirst Ganache accounts:")
for i, acc in enumerate(accounts[:10]):
    print(i, acc)

Connected to Ganache: True
Chain ID: 1337
Number of accounts: 20
Owner Account: 0x465614492F4262B6a7f586c2079Dd6E998272De4

First Ganache accounts:
0 0x465614492F4262B6a7f586c2079Dd6E998272De4
1 0x3a8d7830fecD095682572e09B9905da4B57E7A72
2 0x53E2cd2641435bbCc441cA6FCE21520206fe70f5
3 0xC7912c59d036482807d42976Dc0c145de6159E9c
4 0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4
5 0xE98b036e15318D1bd89A933eED33Bee42692cF43
6 0x65E47B2E5bC2e922A152A77455cC9aaC8b4FE815
7 0x420A8393612C49EB56e714005d5D0ce1ae892791
8 0x93d8D5f552305B281159859d6E960C27dEccBD3b
9 0x8E834E4612A5e78B27Ab9E21B383EE8C2173fCF1


### الخطوة 4
### Cell 5 — Compile + Deploy للعقد الذكي

هذه الخلية:

1. تعمل Compile للعقد الذكي
2. تنشره على Ganache
3. تحفظ ABI
4. تحفظ contract address
5. تحفظ Deployment Log

In [7]:
from datetime import datetime
import json

from solcx import compile_standard, install_solc, set_solc_version

# ============================================================
# Compile + Deploy Smart Contract
# ============================================================

SOLC_VERSION = "0.8.20"

ABI_OUTPUT_PATH = CONTRACT_DIR / "RecommendationDelivery_abi.json"
ADDRESS_OUTPUT_PATH = CONTRACT_DIR / "contract_address.txt"
DEPLOYMENT_LOG_PATH = OUTPUTS_DIR / "04_smart_contract_deployment_log.json"

print("Installing / selecting Solidity compiler:", SOLC_VERSION)
install_solc(SOLC_VERSION)
set_solc_version(SOLC_VERSION)

# قراءة كود العقد الذكي من الملف المحفوظ داخل 01_contract
source_code = SOLIDITY_FILE.read_text(encoding="utf-8")

# ============================================================
# Compile Solidity Contract
# ============================================================

compiled = compile_standard(
    {
        "language": "Solidity",
        "sources": {
            "RecommendationDelivery.sol": {
                "content": source_code
            }
        },
        "settings": {
            "optimizer": {
                "enabled": True,
                "runs": 200
            },
            "outputSelection": {
                "*": {
                    "*": [
                        "abi",
                        "evm.bytecode"
                    ]
                }
            }
        }
    }
)

contract_data = compiled["contracts"]["RecommendationDelivery.sol"]["RecommendationDelivery"]

contract_abi = contract_data["abi"]
contract_bytecode = contract_data["evm"]["bytecode"]["object"]

print("Compilation completed successfully.")

# ============================================================
# Deploy Contract on Ganache
# ============================================================

RecommendationDelivery = w3.eth.contract(
    abi=contract_abi,
    bytecode=contract_bytecode
)

print("Deploying RecommendationDelivery contract...")

deploy_tx_hash = RecommendationDelivery.constructor().transact(
    {
        "from": OWNER_ACCOUNT
    }
)

deploy_receipt = w3.eth.wait_for_transaction_receipt(deploy_tx_hash)

if deploy_receipt.status != 1:
    raise RuntimeError("Contract deployment failed.")

contract_address = deploy_receipt.contractAddress

# ============================================================
# Save ABI and Contract Address
# ============================================================

ABI_OUTPUT_PATH.write_text(
    json.dumps(contract_abi, indent=2),
    encoding="utf-8"
)

ADDRESS_OUTPUT_PATH.write_text(
    contract_address,
    encoding="utf-8"
)

# ============================================================
# Save Deployment Log
# ============================================================

deployment_log = {
    "deployment_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "rpc_url": RPC_URL,
    "chain_id": w3.eth.chain_id,
    "contract_name": "RecommendationDelivery",
    "contract_address": contract_address,
    "deployment_tx_hash": deploy_tx_hash.hex(),
    "owner_account": OWNER_ACCOUNT,
    "solidity_file": str(SOLIDITY_FILE),
    "abi_path": str(ABI_OUTPUT_PATH),
    "contract_address_path": str(ADDRESS_OUTPUT_PATH),
    "solc_version": SOLC_VERSION,
    "ganache_accounts_count": len(accounts)
}

DEPLOYMENT_LOG_PATH.write_text(
    json.dumps(deployment_log, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Contract deployed successfully.")
print("Contract address:", contract_address)
print("Deployment tx:", deploy_tx_hash.hex())
print("ABI saved to:", ABI_OUTPUT_PATH)
print("Contract address saved to:", ADDRESS_OUTPUT_PATH)
print("Deployment log saved to:", DEPLOYMENT_LOG_PATH)

Installing / selecting Solidity compiler: 0.8.20
Compilation completed successfully.
Deploying RecommendationDelivery contract...
Contract deployed successfully.
Contract address: 0xf8FFD2353Fa7FB61fFc25750b6c5B991e2BD966e
Deployment tx: fbb6326f2479e8d943b79be8cc68c7304a079f468db239dd6472771a7b2ee545
ABI saved to: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\01_contract\RecommendationDelivery_abi.json
Contract address saved to: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\01_contract\contract_address.txt
Deployment log saved to: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\03_outputs\04_smart_contract_deployment_log.json


### الخطوة 5
### Cell 6 — تحميل العقد المنشور

In [8]:
with open(ABI_OUTPUT_PATH, "r", encoding="utf-8") as f:
    loaded_abi = json.load(f)

loaded_contract_address = ADDRESS_OUTPUT_PATH.read_text(encoding="utf-8").strip()
loaded_contract_address = Web3.to_checksum_address(loaded_contract_address)

contract = w3.eth.contract(
    address=loaded_contract_address,
    abi=loaded_abi
)

print("Loaded contract at:", loaded_contract_address)

Loaded contract at: 0xf8FFD2353Fa7FB61fFc25750b6c5B991e2BD966e


### الخطوة 6
# Cell 7 — إعدادات الإيصال ومسارات الملفات
عدد التوصيات التي سوف نرسلها هي 5 و نستطيع تغييرها

In [9]:
# ============================================================
# إعدادات إيصال التوصيات
# ============================================================

# 5  => Top-5
# 10 => Top-10
# 20 => Top-20
DELIVERY_TOP_N = 5

# إذا أعدت تنفيذ نفس التجربة على نفس العقد، غيّر النسخة إلى 2 أو 3
DELIVERY_VERSION = 1

DELIVERY_OUTPUT_DIR = OUTPUTS_DIR / f"RECOMMENDATION_DELIVERY_TOP{DELIVERY_TOP_N}_V{DELIVERY_VERSION}"
PAYLOAD_DIR = DELIVERY_OUTPUT_DIR / "payloads"

DELIVERY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("DELIVERY_TOP_N:", DELIVERY_TOP_N)
print("DELIVERY_VERSION:", DELIVERY_VERSION)
print("DELIVERY_OUTPUT_DIR:", DELIVERY_OUTPUT_DIR)


# ============================================================
# ملف القادة الحقيقيين الناتج من مرحلة التصويت
# ============================================================

TRUSTED_LEADERS_SUMMARY_PATH = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\2 التصويت\OP\RUN_20260309_031845\outputs\scenario1_global_friend_distance_sumdist_no_fallback\final\scenario1_global_friend_sumdist_no_fallback_summary.csv"
)

print("\nTrusted leaders file exists:", TRUSTED_LEADERS_SUMMARY_PATH.exists())
print(TRUSTED_LEADERS_SUMMARY_PATH)


# ============================================================
# ملفات التوصيات Top-20 لكل مجتمع
# ============================================================

RECOMMENDATION_FILES = [
    Path(r"D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_003_comm_8_U104_I3510\003_comm_8_U104_I3510_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv"),

    Path(r"D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_006_comm_14_U137_I3673\006_comm_14_U137_I3673_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv"),

    Path(r"D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_023_comm_84_U117_I3855\023_comm_84_U117_I3855_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv"),

    Path(r"D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_026_comm_96_U137_I4033\026_comm_96_U137_I4033_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv"),

    Path(r"D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_028_comm_101_U80_I3034\028_comm_101_U80_I3034_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv"),

    Path(r"D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_045_comm_287_U118_I2606\045_comm_287_U118_I2606_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv"),
]

print("\nChecking recommendation files:")
for p in RECOMMENDATION_FILES:
    print(p.exists(), "=>", p)

DELIVERY_TOP_N: 5
DELIVERY_VERSION: 1
DELIVERY_OUTPUT_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1

Trusted leaders file exists: True
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\2 التصويت\OP\RUN_20260309_031845\outputs\scenario1_global_friend_distance_sumdist_no_fallback\final\scenario1_global_friend_sumdist_no_fallback_summary.csv

Checking recommendation files:
True => D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_003_comm_8_U104_I3510\003_comm_8_U104_I3510_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv
True => D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_006_comm_14_U137_I3673\006_comm_14_U137_I3673_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv
True 

### الخطوة 7
### Cell 8 — تحميل القادة الحقيقيين وربطهم بحسابات Ganache

In [10]:
if not TRUSTED_LEADERS_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Trusted leaders summary not found: {TRUSTED_LEADERS_SUMMARY_PATH}")

leaders_df = pd.read_csv(TRUSTED_LEADERS_SUMMARY_PATH)

required_cols = {"interest_comm_id", "leader"}
missing_cols = required_cols - set(leaders_df.columns)

if missing_cols:
    raise ValueError(f"Missing required columns in trusted leaders summary: {missing_cols}")

leaders_df = leaders_df.copy()
leaders_df = leaders_df.dropna(subset=["interest_comm_id", "leader"])

leaders_df["community_id"] = leaders_df["interest_comm_id"].astype(int)
leaders_df["trusted_node_user_id"] = leaders_df["leader"].astype(int)

optional_cols = []
for c in ["decision_basis", "leader_in_I", "leader_sum_dist", "leader_coverage_count"]:
    if c in leaders_df.columns:
        optional_cols.append(c)

leaders_df = leaders_df[
    ["community_id", "trusted_node_user_id"] + optional_cols
].copy()

unique_leaders = sorted(leaders_df["trusted_node_user_id"].unique().tolist())

# Account[0] محجوز للـ owner
# نبدأ من Account[1] للقادة الحقيقيين
if len(accounts) < len(unique_leaders) + 1:
    raise RuntimeError(
        f"Not enough Ganache accounts. Need at least {len(unique_leaders) + 1}, "
        f"but found {len(accounts)}. Restart Ganache with more accounts."
    )

leader_to_account = {}

for idx, leader_user_id in enumerate(unique_leaders, start=1):
    leader_to_account[int(leader_user_id)] = Web3.to_checksum_address(accounts[idx])

leaders_df["trusted_node_address"] = leaders_df["trusted_node_user_id"].map(leader_to_account)

trusted_nodes_df = leaders_df.copy()

TRUSTED_NODES_OUTPUT_PATH = OUTPUTS_DIR / "05_trusted_nodes_from_voting.csv"

trusted_nodes_df.to_csv(
    TRUSTED_NODES_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Trusted nodes loaded from voting results:")
print(trusted_nodes_df)

print("\nLeader to Ganache account mapping:")
for leader_id, addr in leader_to_account.items():
    print(f"leader user_id={leader_id} -> {addr}")

print("\nSaved trusted-node mapping to:")
print(TRUSTED_NODES_OUTPUT_PATH)

Trusted nodes loaded from voting results:
   community_id  trusted_node_user_id decision_basis  leader_in_I  \
0             8                   378   tie_coverage        False   
1            14                   289         tie_id        False   
2            84                    28   tie_coverage        False   
3            96                    30         tie_id        False   
4           101                    28         tie_id        False   
5           287                   460         tie_id        False   

   leader_sum_dist  leader_coverage_count  \
0              0.0                      3   
1              0.0                      1   
2              0.0                      2   
3              0.0                      1   
4              0.0                      1   
5              0.0                      1   

                         trusted_node_address  
0  0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4  
1  0xC7912c59d036482807d42976Dc0c145de6159E9c  
2  0x3a8d7830f

### الخطوة 8
### Cell 9 — الدوال المساعدة
Cell 9 هي خلية تجهيز فقط، وفائدتها الأساسية أنها تجمع كل الدوال التي تحتاجها المراحل اللاحقة بدل تكرار الكود.
فوائدها باختصار:
1. تربط ملفات التوصيات بالمجتمع الصحيح
تستخرج رقم المجتمع من اسم ملف التوصيات، مثل استخراج المجتمع 8 من اسم الملف الذي يحتوي comm_8.

2. تحدد العقدة الموثوقة الصحيحة لكل مجتمع
تربط كل مجتمع بالقائد الحقيقي Aₖ الناتج من مرحلة التصويت.

3. تحوّل المستخدمين إلى عناوين بلوكشين
لأن العقد الذكي يتعامل مع address وليس مع user_id.

4. تجهز حزمة التوصيات قبل حساب الهاش
تجمع معلومات التوصيات مثل المستخدم، المجتمع، النسخة، عدد التوصيات، والـ nonce.

5. تضمن ثبات صيغة البيانات قبل حساب الهاش
حتى لا يتغير الهاش بسبب اختلاف ترتيب الحقول أو شكل JSON.

6. تحسب Hash التوصيات بطريقة موحدة
نفس طريقة الحساب تُستخدم عند التخزين وعند التحقق.

7. تسجل العقدة الموثوقة داخل العقد الذكي
حتى يعرف العقد أن هذا العنوان هو المخوّل للمجتمع المعني.

8. تسجل أعضاء المجتمع داخل العقد الذكي
حتى لا يستطيع القائد تخزين Hash لتوصيات مستخدم لا ينتمي للمجتمع.

9. تخزن Hash التوصيات من حساب القائد الصحيح
وليس من أي حساب عشوائي في Ganache.

10. تختار Top-N من ملف Top-20
مثل اختيار Top-5 أو Top-10 بدون إعادة حساب التوصيات.

الخلاصة النهائية:
هذه الخلية لا تنفذ الإيصال مباشرة، لكنها تجهز كل الأدوات الضرورية حتى تستطيع الخلايا التالية تسجيل القادة، تسجيل الأعضاء، تخزين الهاشات، وإنشاء ملفات الإيصال والتحقق منها بطريقة منظمة وصحيحة.

In [12]:
def extract_community_id_from_path(path: Path) -> int:
    """
    يستخرج community_id من اسم ملف التوصيات.

    مثال:
    003_comm_8_U104_I3510_top20_per_user_recs_...
    يرجع:
    8
    """
    file_name = path.name
    match = re.search(r"comm_(\d+)_", file_name)

    if not match:
        raise ValueError(f"Cannot extract community_id from file name: {file_name}")

    return int(match.group(1))


def get_trusted_node_address_for_community(community_id: int) -> str:
    """
    يجلب عنوان A_k الحقيقي للمجتمع من ملف القادة.
    """
    row = trusted_nodes_df[trusted_nodes_df["community_id"] == int(community_id)]

    if row.empty:
        raise ValueError(f"No trusted node found for community_id={community_id}")

    return Web3.to_checksum_address(row.iloc[0]["trusted_node_address"])


def get_trusted_node_user_id_for_community(community_id: int) -> int:
    """
    يجلب user_id الخاص بالقائد A_k لذلك المجتمع.
    """
    row = trusted_nodes_df[trusted_nodes_df["community_id"] == int(community_id)]

    if row.empty:
        raise ValueError(f"No trusted node found for community_id={community_id}")

    return int(row.iloc[0]["trusted_node_user_id"])


def user_address_for_dataset_user(user_id: int) -> str:
    """
    يرجع عنوان بلوكشين للمستخدم.

    إذا كان user_id هو قائد موثوق في أي مجتمع:
        نستخدم حساب Ganache المربوط به حتى يستطيع إرسال معاملات عندما يكون A_k.

    إذا كان user_id مستخدمًا عاديًا:
        نولّد له عنوانًا تمثيليًا deterministic.
        هذا العنوان لا يرسل معاملات، لكنه يستخدم كهوية مستلمة للتوصيات.
    """
    user_id = int(user_id)

    if user_id in leader_to_account:
        return Web3.to_checksum_address(leader_to_account[user_id])

    digest = w3.keccak(text=f"douban-user-{user_id}").hex()
    address = "0x" + digest[-40:]
    return Web3.to_checksum_address(address)


def build_recommendation_payload_for_hash(
    user_address: str,
    community_id: int,
    version: int,
    top_n: int,
    recommendations: list,
    nonce: str
) -> dict:
    """
    هذه الحزمة هي الأساس الذي نحسب منه الهاش.
    يجب أن تكون مطابقة بين مرحلة التخزين ومرحلة التحقق.
    """
    return {
        "user_address": Web3.to_checksum_address(user_address),
        "community_id": int(community_id),
        "version": int(version),
        "top_n": int(top_n),
        "recommendations": recommendations,
        "nonce": str(nonce)
    }


def canonical_json(data: dict) -> str:
    """
    يحوّل البيانات إلى JSON ثابت الترتيب.
    هذا ضروري حتى يكون الهاش قابلًا للإعادة بنفس النتيجة.
    """
    return json.dumps(
        data,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    )


def compute_recommendation_hash(
    user_address: str,
    community_id: int,
    version: int,
    top_n: int,
    recommendations: list,
    nonce: str
):
    """
    يحسب keccak256 hash لحزمة التوصيات.
    """
    data = build_recommendation_payload_for_hash(
        user_address=user_address,
        community_id=community_id,
        version=version,
        top_n=top_n,
        recommendations=recommendations,
        nonce=nonce
    )

    message = canonical_json(data)

    return w3.keccak(text=message)


def ensure_authorized_node(community_id: int):
    """
    يسجل A_k الحقيقي لهذا المجتمع داخل العقد الذكي.
    """
    trusted_node_address = get_trusted_node_address_for_community(community_id)

    current_node = contract.functions.authorizedNode(int(community_id)).call()

    if current_node.lower() == trusted_node_address.lower():
        print(f"[A_k] Already set for community {community_id}: {trusted_node_address}")
        return trusted_node_address

    tx_hash = contract.functions.setAuthorizedNode(
        int(community_id),
        trusted_node_address
    ).transact(
        {
            "from": OWNER_ACCOUNT
        }
    )

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    if receipt.status != 1:
        raise RuntimeError(f"Failed to set A_k for community {community_id}")

    print(f"[A_k] Set for community {community_id}: {trusted_node_address}")

    return trusted_node_address


def register_community_members_onchain(community_id: int, member_addresses: list, batch_size: int = 100):
    """
    يسجل أعضاء المجتمع داخل العقد الذكي.

    السبب:
    حتى لا تستطيع A_k تخزين Hash توصيات لمستخدم غير عضو في مجتمعها.
    """
    member_addresses = [Web3.to_checksum_address(x) for x in member_addresses]
    unique_members = sorted(set(member_addresses))

    print(f"[MEMBERS] Registering {len(unique_members)} members for community {community_id}")

    tx_hashes = []

    for start in range(0, len(unique_members), batch_size):
        batch = unique_members[start:start + batch_size]

        tx_hash = contract.functions.registerCommunityMembers(
            int(community_id),
            batch
        ).transact(
            {
                "from": OWNER_ACCOUNT,
                "gas": 6_000_000
            }
        )

        receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

        if receipt.status != 1:
            raise RuntimeError(
                f"Failed to register members for community={community_id}, "
                f"batch_start={start}"
            )

        tx_hashes.append(tx_hash.hex())

        print(
            f"[MEMBERS] community={community_id}, "
            f"batch={start // batch_size + 1}, "
            f"tx={tx_hash.hex()}"
        )

    count = contract.functions.communityMemberCount(int(community_id)).call()
    print(f"[MEMBERS] On-chain member count for community {community_id}: {count}")

    return tx_hashes


def store_hash_onchain(
    community_id: int,
    user_address: str,
    version: int,
    top_n: int,
    recommendation_hash
):
    """
    يخزن الهاش في العقد الذكي.
    المعاملة يجب أن ترسل من A_k الحقيقي لذلك المجتمع.
    """
    trusted_node_address = get_trusted_node_address_for_community(community_id)

    exists = contract.functions.recordExists(
        int(community_id),
        Web3.to_checksum_address(user_address),
        int(version),
        int(top_n)
    ).call()

    if exists:
        return None, trusted_node_address

    tx_hash = contract.functions.storeRecommendationHash(
        int(community_id),
        Web3.to_checksum_address(user_address),
        int(version),
        int(top_n),
        recommendation_hash
    ).transact(
        {
            "from": trusted_node_address,
            "gas": 500_000
        }
    )

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    if receipt.status != 1:
        raise RuntimeError(
            f"Failed to store hash for community={community_id}, "
            f"user={user_address}, version={version}, topN={top_n}"
        )

    return tx_hash.hex(), trusted_node_address


def build_user_topn_recommendations(recs_df: pd.DataFrame, user_id: int, top_n: int) -> list:
    """
    يأخذ توصيات مستخدم واحد من ملف Top-20 ويختار Top-N.

    نحتفظ بـ:
    - rank
    - item

    لا نضع PredScore داخل حزمة الإيصال لتقليل حساسية الهاش تجاه تقريب الأرقام.
    """
    user_recs = recs_df[recs_df["user"].astype(int) == int(user_id)].copy()

    if user_recs.empty:
        return []

    user_recs = user_recs.sort_values("rank")
    user_recs = user_recs[user_recs["rank"] <= top_n]

    recommendations = []

    for _, row in user_recs.iterrows():
        recommendations.append(
            {
                "rank": int(row["rank"]),
                "item": int(row["item"])
            }
        )

    return recommendations

print("[OK] Helper functions were defined successfully.")
print("Available helper functions:")
print("- extract_community_id_from_path")
print("- get_trusted_node_address_for_community")
print("- get_trusted_node_user_id_for_community")
print("- user_address_for_dataset_user")
print("- compute_recommendation_hash")
print("- ensure_authorized_node")
print("- register_community_members_onchain")
print("- store_hash_onchain")
print("- build_user_topn_recommendations")

[OK] Helper functions were defined successfully.
Available helper functions:
- extract_community_id_from_path
- get_trusted_node_address_for_community
- get_trusted_node_user_id_for_community
- user_address_for_dataset_user
- compute_recommendation_hash
- ensure_authorized_node
- register_community_members_onchain
- store_hash_onchain
- build_user_topn_recommendations


### الخطوة 9
### Cell 10 — تسجيل A_k وأعضاء المجتمعات داخل العقد

هذه الخلية تسجل داخل العقد:

community_id → A_k
community_id → members

In [13]:
registration_records = []
membership_records = []

for rec_path in RECOMMENDATION_FILES:

    if not rec_path.exists():
        print("[WARN] Recommendation file not found, skipping:", rec_path)
        continue

    community_id = extract_community_id_from_path(rec_path)

    print("\n" + "=" * 100)
    print("[REGISTER] community_id:", community_id)
    print("[FILE]", rec_path)
    print("=" * 100)

    # 1. تسجيل A_k الحقيقي للمجتمع
    trusted_node_address = ensure_authorized_node(community_id)
    trusted_node_user_id = get_trusted_node_user_id_for_community(community_id)

    # 2. استخراج أعضاء المجتمع من ملف التوصيات
    recs_df = pd.read_csv(rec_path)

    if "user" not in recs_df.columns:
        raise ValueError(f"Column 'user' not found in {rec_path}")

    users = sorted(recs_df["user"].dropna().astype(int).unique().tolist())
    member_addresses = [user_address_for_dataset_user(u) for u in users]

    # 3. تسجيل أعضاء المجتمع داخل العقد
    member_tx_hashes = register_community_members_onchain(
        community_id=community_id,
        member_addresses=member_addresses,
        batch_size=100
    )

    registration_records.append(
        {
            "community_id": int(community_id),
            "trusted_node_user_id": int(trusted_node_user_id),
            "trusted_node_address": trusted_node_address,
            "member_count_input": int(len(users)),
            "member_count_onchain": int(contract.functions.communityMemberCount(int(community_id)).call()),
            "member_registration_txs": "|".join(member_tx_hashes),
            "source_recommendation_file": str(rec_path)
        }
    )

    for u, addr in zip(users, member_addresses):
        membership_records.append(
            {
                "community_id": int(community_id),
                "user_id": int(u),
                "user_address": addr,
                "trusted_node_user_id": int(trusted_node_user_id),
                "trusted_node_address_for_community": trusted_node_address
            }
        )

registration_df = pd.DataFrame(registration_records)
membership_df = pd.DataFrame(membership_records)

REGISTRATION_OUTPUT_PATH = OUTPUTS_DIR / "06_onchain_authorized_nodes_and_members_registration.csv"
MEMBERSHIP_OUTPUT_PATH = OUTPUTS_DIR / "07_community_membership_addresses_registered_onchain.csv"

registration_df.to_csv(
    REGISTRATION_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

membership_df.to_csv(
    MEMBERSHIP_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved registration summary to:")
print(REGISTRATION_OUTPUT_PATH)

print("\nSaved community membership address mapping to:")
print(MEMBERSHIP_OUTPUT_PATH)

registration_df


[REGISTER] community_id: 8
[FILE] D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_003_comm_8_U104_I3510\003_comm_8_U104_I3510_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv
[A_k] Set for community 8: 0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4
[MEMBERS] Registering 104 members for community 8
[MEMBERS] community=8, batch=1, tx=548850dd041d66cc9d98593900759b4af76e8e0b5953440f53abce14961bb64c
[MEMBERS] community=8, batch=2, tx=488f42dda8c48733150be07c711a9483c9c03238bb8db98ce485a9673a99845d
[MEMBERS] On-chain member count for community 8: 104

[REGISTER] community_id: 14
[FILE] D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_006_comm_14_U137_I3673\006_comm_14_U137_I3673_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv
[A_k] Set for community 14: 0xC7912c59d03648

,community_id,trusted_node_user_id,trusted_node_address,member_count_input,member_count_onchain,member_registration_txs,source_recommendation_file
0,8,378,0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4,104,104,548850dd041d66cc9d98593900759b4af76e8e0b595344...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
1,14,289,0xC7912c59d036482807d42976Dc0c145de6159E9c,137,137,40d5b7c89982c5d274392f915bbfc50025880b9f3bda0f...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
2,84,28,0x3a8d7830fecD095682572e09B9905da4B57E7A72,117,117,e94628c1c1f251bd172f0f866c0aff8262ae7d953d1d42...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
3,96,30,0x53E2cd2641435bbCc441cA6FCE21520206fe70f5,137,137,407372a05d2a0a3664f9cb12253b55da0091dcf53d9a84...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
4,101,28,0x3a8d7830fecD095682572e09B9905da4B57E7A72,80,80,005d3087812faa3d019e119f83fec5ea44c235e698d748...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
5,287,460,0xE98b036e15318D1bd89A933eED33Bee42692cF43,118,118,25de52dfa3a828f94b16bc5997d99df6e811988cf8f884...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...


### الخطوة 10
### Cell 11 — تنفيذ إيصال التوصيات وتخزين Hash

هذه الخلية هي قلب مرحلة الإيصال.

In [14]:
 all_delivery_records = []
skipped_existing = 0

for rec_path in RECOMMENDATION_FILES:

    if not rec_path.exists():
        print("[WARN] Recommendation file not found, skipping:", rec_path)
        continue

    community_id = extract_community_id_from_path(rec_path)

    print("\n" + "=" * 100)
    print("[DELIVERY] community_id:", community_id)
    print("[FILE]", rec_path)
    print("=" * 100)

    trusted_node_address = get_trusted_node_address_for_community(community_id)
    trusted_node_user_id = get_trusted_node_user_id_for_community(community_id)

    recs_df = pd.read_csv(rec_path)

    required_cols = {"user", "item", "rank"}
    missing_cols = required_cols - set(recs_df.columns)

    if missing_cols:
        raise ValueError(f"Missing required columns in {rec_path}: {missing_cols}")

    users = sorted(recs_df["user"].dropna().astype(int).unique().tolist())

    community_payload_dir = PAYLOAD_DIR / f"community_{community_id}"
    community_payload_dir.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Number of users in community {community_id}: {len(users)}")
    print(f"[INFO] A_k user_id for community {community_id}: {trusted_node_user_id}")
    print(f"[INFO] A_k address for community {community_id}: {trusted_node_address}")

    for user_id in users:

        user_address = user_address_for_dataset_user(user_id)

        recommendations = build_user_topn_recommendations(
            recs_df=recs_df,
            user_id=user_id,
            top_n=DELIVERY_TOP_N
        )

        if len(recommendations) == 0:
            print(f"[WARN] Empty recommendations for user={user_id}, community={community_id}")
            continue

        nonce = secrets.token_hex(16)

        recommendation_hash = compute_recommendation_hash(
            user_address=user_address,
            community_id=community_id,
            version=DELIVERY_VERSION,
            top_n=DELIVERY_TOP_N,
            recommendations=recommendations,
            nonce=nonce
        )

        tx_hash, issuer_address = store_hash_onchain(
            community_id=community_id,
            user_address=user_address,
            version=DELIVERY_VERSION,
            top_n=DELIVERY_TOP_N,
            recommendation_hash=recommendation_hash
        )

        if tx_hash is None:
            skipped_existing += 1
            print(
                f"[SKIP] Existing record: community={community_id}, "
                f"user={user_id}, version={DELIVERY_VERSION}, topN={DELIVERY_TOP_N}"
            )
            continue

        payload = {
            "community_id": int(community_id),
            "user_id": int(user_id),
            "user_address": user_address,
            "version": int(DELIVERY_VERSION),
            "top_n": int(DELIVERY_TOP_N),
            "recommendations": recommendations,
            "nonce": nonce,
            "recommendation_hash": recommendation_hash.hex(),
            "trusted_node_user_id": int(trusted_node_user_id),
            "issuer_authorized_node": issuer_address,
            "tx_hash": tx_hash,
            "source_recommendation_file": str(rec_path)
        }

        payload_path = community_payload_dir / (
            f"user_{user_id}_community_{community_id}_v{DELIVERY_VERSION}_top{DELIVERY_TOP_N}_payload.json"
        )

        with open(payload_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)

        all_delivery_records.append(
            {
                "community_id": int(community_id),
                "user_id": int(user_id),
                "user_address": user_address,
                "version": int(DELIVERY_VERSION),
                "top_n": int(DELIVERY_TOP_N),
                "recommendation_hash": recommendation_hash.hex(),
                "trusted_node_user_id": int(trusted_node_user_id),
                "issuer_authorized_node": issuer_address,
                "tx_hash": tx_hash,
                "payload_path": str(payload_path),
                "source_recommendation_file": str(rec_path)
            }
        )

    print(f"[DONE] Community {community_id} completed.")

delivery_summary_df = pd.DataFrame(all_delivery_records)

DELIVERY_SUMMARY_PATH = DELIVERY_OUTPUT_DIR / f"08_onchain_delivery_records_top{DELIVERY_TOP_N}_v{DELIVERY_VERSION}.csv"

delivery_summary_df.to_csv(
    DELIVERY_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 100)
print("[SUMMARY] Delivery preparation completed.")
print("[SUMMARY] New delivery records:", len(delivery_summary_df))
print("[SUMMARY] Skipped existing records:", skipped_existing)
print("[SUMMARY] Summary CSV:", DELIVERY_SUMMARY_PATH)
print("[SUMMARY] Payload directory:", PAYLOAD_DIR)
print("=" * 100)

delivery_summary_df.head()


[DELIVERY] community_id: 8
[FILE] D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_003_comm_8_U104_I3510\003_comm_8_U104_I3510_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv
[INFO] Number of users in community 8: 104
[INFO] A_k user_id for community 8: 378
[INFO] A_k address for community 8: 0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4
[DONE] Community 8 completed.

[DELIVERY] community_id: 14
[FILE] D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMUNITIES_DETERMINISTIC\RUN_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20_006_comm_14_U137_I3673\006_comm_14_U137_I3673_top20_per_user_recs_VR14b_COMM_DET_LN0d2_NEG2_EMB64_POS3train4test_BPR_EVAL20.csv
[INFO] Number of users in community 14: 137
[INFO] A_k user_id for community 14: 289
[INFO] A_k address for community 14: 0xC7912c59d036482807d42976Dc0c145de6159E9c
[DONE] Community 14 completed.

[DELIVERY] community_id: 84


,community_id,user_id,user_address,version,top_n,recommendation_hash,trusted_node_user_id,issuer_authorized_node,tx_hash,payload_path,source_recommendation_file
0,8,13,0xdA6a4AB9edB3fF50fE8D1C3A53BE622617b590F0,1,5,743f4404ad19d9f005ad2c647f62c5441fde0d00e3d3dc...,378,0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4,c5104c95f0e7540ca419bf4da6bb5cef9af819f8877559...,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
1,8,22,0xd08f680834063095f90EebAcF3a63FB53C5c6163,1,5,e81d61fc7d672356240dacce286c4f16d9d4039e9820bb...,378,0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4,7865ba8768d5c16ed96c9245c89a4605e1193e5c9dfb19...,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
2,8,53,0xf01bab8b9b26C0F62Baaa845E7c0EC76BE32265d,1,5,2cbf7c17be289ffc240b321698fccb8604c32cfa53f818...,378,0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4,9ec3fe2c811b4a2f332071b9f8d0eb519e54c898e9d967...,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
3,8,105,0x5AEd1102D75a352B120c37D8d695A348788ed44B,1,5,c1fc9ad77687dd712c33f8418dd22e1aa0abb3c088afe2...,378,0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4,c07ee054bf7e07420a44587dadd6530dcce2110d2e95e4...,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...
4,8,124,0x8462c1EE0B031cDe7A78782F533A6d014E051C79,1,5,382aad802d16fbeda5946be5dd1ad5675661fa01e5dd87...,378,0xf1667a0c3907c5Ce4b53E38490E1Db2A4357D9d4,24532aa46e6dcab76c6813f218ac1e728b965aaab9d807...,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,D:\PHD\results\13 Uplift\DMF_GLOBAL_ONLY_COMMU...


### الخطوة 11
### Cell 12 — التحقق من Payload واحد

In [15]:
payload_files = sorted(PAYLOAD_DIR.rglob("*_payload.json"))

if len(payload_files) == 0:
    raise RuntimeError("No payload files found. شغّل خلية الإيصال أولًا أو غيّر DELIVERY_VERSION.")

PAYLOAD_PATH = payload_files[0]

print("Selected payload:")
print(PAYLOAD_PATH)


def verify_payload(payload_path: Path):
    """
    تتحقق من payload واحد:
    - تقرأ التوصيات والـ nonce.
    - تعيد حساب الهاش محليًا.
    - تقرأ الهاش من العقد الذكي.
    - تقارن النتيجة.
    """
    with open(payload_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    community_id = int(payload["community_id"])
    user_address = Web3.to_checksum_address(payload["user_address"])
    version = int(payload["version"])
    top_n = int(payload["top_n"])
    recommendations = payload["recommendations"]
    nonce = str(payload["nonce"])

    local_hash = compute_recommendation_hash(
        user_address=user_address,
        community_id=community_id,
        version=version,
        top_n=top_n,
        recommendations=recommendations,
        nonce=nonce
    )

    stored_hash = contract.functions.getRecommendationHash(
        community_id,
        user_address,
        version,
        top_n
    ).call()

    contract_result = contract.functions.verifyRecommendationHash(
        community_id,
        user_address,
        version,
        top_n,
        local_hash
    ).call()

    record = contract.functions.getRecommendationRecord(
        community_id,
        user_address,
        version,
        top_n
    ).call()

    print("=" * 100)
    print("[PAYLOAD]", payload_path)
    print("[COMMUNITY]", community_id)
    print("[USER ADDRESS]", user_address)
    print("[VERSION]", version)
    print("[TOP_N]", top_n)
    print("[LOCAL HASH]", local_hash.hex())
    print("[STORED HASH]", stored_hash.hex())
    print("[CONTRACT VERIFY]", contract_result)
    print("[ISSUER FROM CONTRACT]", record[1])

    if local_hash == stored_hash and contract_result:
        print("[RESULT] ACCEPTED: The received recommendations match the on-chain hash.")
        result = True
    else:
        print("[RESULT] REJECTED: The received recommendations do not match the on-chain hash.")
        result = False

    print("=" * 100)

    return result


verify_payload(PAYLOAD_PATH)

Selected payload:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1\payloads\community_101\user_1098_community_101_v1_top5_payload.json
[PAYLOAD] D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1\payloads\community_101\user_1098_community_101_v1_top5_payload.json
[COMMUNITY] 101
[USER ADDRESS] 0xB714535e97D122704bF4cF9b9b29ce3C18287a71
[VERSION] 1
[TOP_N] 5
[LOCAL HASH] 5899aec681b8ab287ef50148367c3af771c2a7364566da8e23e77617b58c5a47
[STORED HASH] 5899aec681b8ab287ef50148367c3af771c2a7364566da8e23e77617b58c5a47
[CONTRACT VERIFY] True
[ISSUER FROM CONTRACT] 0x3a8d7830fecD095682572e09B9905da4B57E7A72
[RESULT] ACCEPTED: The received recommendations match the on-chain hash.


True

### 12 الخطوة 
### Cell 13 — التحقق من كل ملفات Payloads

In [16]:
verification_results = []

for p in sorted(PAYLOAD_DIR.rglob("*_payload.json")):
    with open(p, "r", encoding="utf-8") as f:
        payload = json.load(f)

    community_id = int(payload["community_id"])
    user_address = Web3.to_checksum_address(payload["user_address"])
    version = int(payload["version"])
    top_n = int(payload["top_n"])
    recommendations = payload["recommendations"]
    nonce = str(payload["nonce"])

    local_hash = compute_recommendation_hash(
        user_address=user_address,
        community_id=community_id,
        version=version,
        top_n=top_n,
        recommendations=recommendations,
        nonce=nonce
    )

    contract_result = contract.functions.verifyRecommendationHash(
        community_id,
        user_address,
        version,
        top_n,
        local_hash
    ).call()

    verification_results.append(
        {
            "payload_path": str(p),
            "community_id": community_id,
            "user_address": user_address,
            "version": version,
            "top_n": top_n,
            "local_hash": local_hash.hex(),
            "verified": bool(contract_result)
        }
    )

verification_df = pd.DataFrame(verification_results)

VERIFICATION_RESULTS_PATH = DELIVERY_OUTPUT_DIR / f"09_verification_results_top{DELIVERY_TOP_N}_v{DELIVERY_VERSION}.csv"

verification_df.to_csv(
    VERIFICATION_RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Verification results saved to:")
print(VERIFICATION_RESULTS_PATH)

print("\nVerification summary:")
print(verification_df["verified"].value_counts(dropna=False))

verification_df.head()

Verification results saved to:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1\09_verification_results_top5_v1.csv

Verification summary:
verified
True    693
Name: count, dtype: int64


,payload_path,community_id,user_address,version,top_n,local_hash,verified
0,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,101,0xB714535e97D122704bF4cF9b9b29ce3C18287a71,1,5,5899aec681b8ab287ef50148367c3af771c2a7364566da...,True
1,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,101,0xE16D105087E40E151c8cD2A5F0C990131674d721,1,5,f55f9add58ec7f91b89e2fe88bd94cccbb57117fd965b4...,True
2,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,101,0x8DA79eB26414B6cB476275E037191d360125C9B8,1,5,ca29fe61a4e530828371f7994a5008b427c571ba00c858...,True
3,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,101,0xb62f69116f6a5787f90c0F9B19A6909ADF9ACc9A,1,5,d6f0abdf9094ad15317a99212d4a741a05641e61775db4...,True
4,D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Do...,101,0x2AF365a5dC124422EB26DE04843347e488d39D2a,1,5,117c32bdd247c32e6cad1f6f77d1a2e5bd97d7e492f2c7...,True


### الخطوة 13
### Cell 14 — اختبار تلاعب اختياري

هذا الاختبار لا يغيّر ملفاتك. فقط يغيّر توصية داخل الذاكرة ليتأكد أن العقد يرفضها.

In [17]:
test_payload_path = payload_files[0]

with open(test_payload_path, "r", encoding="utf-8") as f:
    test_payload = json.load(f)

print("Original first recommendation:")
print(test_payload["recommendations"][0])

# تلاعب داخل الذاكرة فقط
tampered_payload = json.loads(json.dumps(test_payload))
tampered_payload["recommendations"][0]["item"] = int(tampered_payload["recommendations"][0]["item"]) + 999999

tampered_hash = compute_recommendation_hash(
    user_address=tampered_payload["user_address"],
    community_id=tampered_payload["community_id"],
    version=tampered_payload["version"],
    top_n=tampered_payload["top_n"],
    recommendations=tampered_payload["recommendations"],
    nonce=tampered_payload["nonce"]
)

tampered_result = contract.functions.verifyRecommendationHash(
    int(tampered_payload["community_id"]),
    Web3.to_checksum_address(tampered_payload["user_address"]),
    int(tampered_payload["version"]),
    int(tampered_payload["top_n"]),
    tampered_hash
).call()

print("Tampered hash:", tampered_hash.hex())
print("Contract verification after tampering:", tampered_result)

if not tampered_result:
    print("Expected result: REJECTED because the recommendation content was modified.")
else:
    print("Unexpected result: check the hashing logic.")

Original first recommendation:
{'rank': 1, 'item': 680}
Tampered hash: 82a986962a9c64e4188700fcb8333b79452134bf8d23201949eaafb3330d1d35
Contract verification after tampering: False
Expected result: REJECTED because the recommendation content was modified.
